In [2]:
# %% 0) Setup: imports, paths, helpers
from pathlib import Path
import os, shutil, random, subprocess
import cv2, numpy as np
from tqdm import tqdm

# ---- EDIT THIS: your video path ----
RAW_VIDEO_PATH = r"C:\Users\haria\Desktop\PMK Final report\AAM, EBCA, FM 100\DJI_0415-1.MP4"

# Project root
ROOT       = Path.cwd() / "datasets" / "APPLE"
ROOT.mkdir(parents=True, exist_ok=True)

# Targets
TARGET_FPS = 80
MAX_FRAMES = 4000      # aim to save up to ~4000 frames at 80 fps
PICK_COUNT = 800       # select exactly 800 frames for augmentation

# Folders
FR80_DIR   = ROOT / "frames_80fps_exact"  # all 80 fps frames (up to 4000)
FR800_DIR  = ROOT / "frames_800"          # exactly 800 chosen
AUG800_DIR = ROOT / "aug_800"             # augmented versions of those 800
IM_OUT     = ROOT / "images"              # YOLO images/{train,val,test}
LAB_OUT    = ROOT / "labels"              # YOLO labels/{train,val,test}
LABELS_RAW = ROOT / "labels_raw"          # put your manual YOLO .txt here if you have

for d in [FR80_DIR, FR800_DIR, AUG800_DIR,
          IM_OUT/"train", IM_OUT/"val", IM_OUT/"test",
          LAB_OUT/"train", LAB_OUT/"val", LAB_OUT/"test",
          LABELS_RAW]:
    d.mkdir(parents=True, exist_ok=True)

def save_img(path: Path, img):
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), img)

def have_ffmpeg() -> bool:
    from shutil import which
    return which("ffmpeg") is not None

def run_cmd(cmd: list) -> bool:
    print("[cmd]", " ".join(cmd))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    return p.returncode == 0


KeyboardInterrupt: 

In [ ]:
# %% 2) Extract frames at EXACT 80 FPS
import cv2, numpy as np
from pathlib import Path

RAW_VIDEO_PATH = Path(r"C:\Users\haria\Desktop\PMK Final report\AAM, EBCA, FM 100\DJI_0415-1.MP4")
TARGET_FPS = 80

# ✅ Define output folder for frames
FRAMES = Path(r"d:\omg2\datasets\APPLE\frames_80fps_exact")
FRAMES.mkdir(parents=True, exist_ok=True)

def extract_frames_exact_fps(video_path: Path, out_dir: Path, target_fps: int = 80, overwrite: bool = False):
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(list(out_dir.glob("*.jpg")) + list(out_dir.glob("*.png")))
    if existing and not overwrite:
        print(f"[extract] Found {len(existing)} frames; skipping (overwrite=True to force).")
        return len(existing), None, None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if not src_fps or src_fps <= 0:
        dur = total_frames / 30.0 if total_frames else 0
        src_fps = total_frames / dur if dur else 30.0

    duration = total_frames / src_fps if src_fps > 0 else 0
    if duration <= 0:
        raise RuntimeError("Video duration invalid (0).")

    ts = np.arange(0, duration, 1.0 / float(target_fps))
    if len(ts) == 0 or ts[-1] > duration:
        ts = ts[ts < duration]
    frame_indices = np.clip(np.rint(ts * src_fps).astype(int), 0, max(total_frames - 1, 0))

    saved = 0
    for i, fidx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ok, frame = cap.read()
        if not ok:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(int(fidx) - 1, 0))
            ok, frame = cap.read()
            if not ok:
                continue
        out_path = out_dir / f"{saved:06d}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved += 1
        if saved == 1 or saved % 200 == 0:
            print(f"[extract] {saved}/{len(frame_indices)}")

    cap.release()
    print(f"[extract] Done. Saved {saved} frames @ {target_fps} fps (src ~{src_fps:.3f} fps, {width}x{height}).")
    return saved, src_fps, duration

# ✅ Now call with defined FRAMES
saved_frames, src_fps, duration = extract_frames_exact_fps(RAW_VIDEO_PATH, FRAMES, target_fps=TARGET_FPS, overwrite=False)
print(f"[extract] frames_saved={saved_frames}, src_fps≈{src_fps}, duration≈{duration:.2f}s")


[extract] 1/4578
[extract] 200/4578
[extract] 400/4578
[extract] 600/4578
[extract] 800/4578
[extract] 1000/4578
[extract] 1200/4578
[extract] 1400/4578
[extract] 1600/4578
[extract] 1800/4578
[extract] 2000/4578
[extract] 2200/4578
[extract] 2400/4578
[extract] 2600/4578
[extract] 2800/4578
[extract] 3000/4578
[extract] 3200/4578
[extract] 3400/4578
[extract] 3600/4578
[extract] 3800/4578
[extract] 4000/4578
[extract] 4200/4578
[extract] 4400/4578
[extract] Done. Saved 4578 frames @ 80 fps (src ~29.970 fps, 1920x1080).
[extract] frames_saved=4578, src_fps≈29.97002997002997, duration≈57.22s


In [ ]:
# %% 2) Pick EXACTLY 800 frames uniformly from the 80-fps dump
all_80 = sorted(FR80_DIR.glob("*.jpg"))
assert len(all_80) > 0, f"No frames found in {FR80_DIR}"

TARGET = PICK_COUNT
if len(all_80) <= TARGET:
    selected = all_80
else:
    step = len(all_80) / TARGET
    idxs = [int(i * step) for i in range(TARGET)]
    # resolve duplicates if any
    used, fixed = set(), []
    for idx in idxs:
        while idx in used:
            idx = min(idx + 1, len(all_80) - 1)
        used.add(idx)
        fixed.append(idx)
    selected = [all_80[i] for i in fixed]

# copy to FR800_DIR
for p in tqdm(selected, desc="Copy 800"):
    shutil.copy2(p, FR800_DIR / p.name)

print(f"[select] {len(selected)} frames copied to {FR800_DIR}")


Copy 800: 100%|██████████| 800/800 [00:02<00:00, 299.73it/s]

[select] 800 frames copied to d:\omg2\datasets\APPLE\frames_800


In [ ]:
# %% 3) Augment those 800 frames (native size). Your full list = 10 variants per image
def adjust_brightness(img, factor):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 2] = np.clip(hsv[..., 2] * factor, 0, 255)
    hsv = hsv.astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def rotate_cardinal(img, angle):
    if angle == 90:  return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180: return cv2.rotate(img, cv2.ROTATE_180)
    if angle == 270: return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError("angle must be 90/180/270")

def rotate_random(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))

def motion_blur(img, ksize=7):
    k = np.zeros((ksize, ksize), np.float32)
    k[ksize//2, :] = 1.0 / ksize
    return cv2.filter2D(img, -1, k)

def gaussian_noise(img, sigma=12):
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    out = img.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)

imgs = sorted(FR800_DIR.glob("*.jpg"))
assert len(imgs) == PICK_COUNT or len(imgs) > 0, f"No images in {FR800_DIR}"

made = 0
for p in tqdm(imgs, desc="Augment 800"):
    img  = cv2.imread(str(p))
    stem = p.stem

    # ✅ Horizontal & Vertical Mirroring
    save_img(AUG800_DIR / f"{stem}_hflip.jpg",  cv2.flip(img, 1))
    save_img(AUG800_DIR / f"{stem}_vflip.jpg",  cv2.flip(img, 0))

    # ✅ Brightness Enhancement & Reduction
    save_img(AUG800_DIR / f"{stem}_bup.jpg",    adjust_brightness(img, 1.2))
    save_img(AUG800_DIR / f"{stem}_bdown.jpg",  adjust_brightness(img, 0.8))

    # ✅ Rotations (90°, 180°, 270°, random small)
    save_img(AUG800_DIR / f"{stem}_r90.jpg",    rotate_cardinal(img, 90))
    save_img(AUG800_DIR / f"{stem}_r180.jpg",   rotate_cardinal(img, 180))
    save_img(AUG800_DIR / f"{stem}_r270.jpg",   rotate_cardinal(img, 270))
    save_img(AUG800_DIR / f"{stem}_rrand.jpg",  rotate_random(img, random.uniform(-12, 12)))

    # ✅ Motion Blur
    save_img(AUG800_DIR / f"{stem}_mblur.jpg",  motion_blur(img, 7))

    # ✅ Gaussian Noise
    save_img(AUG800_DIR / f"{stem}_gnoise.jpg", gaussian_noise(img, 12))

    made += 10

print(f"[augment] originals: {len(imgs)} | aug created: {made} | total now: {len(imgs)+made}")


Augment 800: 100%|██████████| 800/800 [05:44<00:00,  2.32it/s]

[augment] originals: 800 | aug created: 8000 | total now: 8800


In [ ]:
# %% 4) Counts after augmentation
n_80   = len(list(FR80_DIR.glob("*.jpg")))
n_800  = len(list(FR800_DIR.glob("*.jpg")))
n_aug  = len(list(AUG800_DIR.glob("*.jpg")))
print(f"[counts] 80fps_total:{n_80} | picked_800:{n_800} | aug_from_800:{n_aug} | grand:{n_800+n_aug}")


[counts] 80fps_total:4578 | picked_800:800 | aug_from_800:8000 | grand:8800


In [ ]:
# %% 5) Split into train/val/test (70/10/20), using 800 originals + their augmentations
random.seed(42)
POOL = sorted(FR800_DIR.glob("*.jpg")) + sorted(AUG800_DIR.glob("*.jpg"))
print(f"[pool] total images considered: {len(POOL)}")

random.shuffle(POOL)
N = len(POOL)
n_train = int(0.7 * N)
n_val   = int(0.1 * N)

train_set = POOL[:n_train]
val_set   = POOL[n_train:n_train+n_val]
test_set  = POOL[n_train+n_val:]

def copy_with_label(img_path: Path, split: str):
    # copy image
    shutil.copy2(img_path, IM_OUT/split/img_path.name)
    # copy label if you already created it in labels_raw; else empty placeholder
    lbl_in  = LABELS_RAW / (img_path.stem + ".txt")
    lbl_out = LAB_OUT/split/(img_path.stem + ".txt")
    if lbl_in.exists():
        shutil.copy2(lbl_in, lbl_out)
    else:
        lbl_out.write_text("")

for p in train_set: copy_with_label(p, "train")
for p in val_set:   copy_with_label(p, "val")
for p in test_set:  copy_with_label(p, "test")

print(f"[split] train={len(train_set)} | val={len(val_set)} | test={len(test_set)}")
print(f"[out] images -> {IM_OUT} | labels -> {LAB_OUT}")


[pool] total images considered: 8800
[split] train=6160 | val=880 | test=1760
[out] images -> d:\omg2\datasets\APPLE\images | labels -> d:\omg2\datasets\APPLE\labels


In [ ]:
# custom_layers.py
import torch, torch.nn as nn, torch.nn.functional as F

class SiLU(nn.SiLU): pass

class AAM(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, r=16, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, k//2, bias=False)
        self.bn   = nn.BatchNorm2d(c2)
        self.act  = SiLU(inplace=True) if act else nn.Identity()
        self.se   = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c2, max(1, c2//r), 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(max(1, c2//r), c2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        y = self.act(self.bn(self.conv(x)))
        y = y * self.se(y)
        return y

class EBCA(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, k//2, bias=False)
        self.bn   = nn.BatchNorm2d(c2)
        self.act  = SiLU(inplace=True) if act else nn.Identity()
        self.satt = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        y = self.act(self.bn(self.conv(x)))
        avg = torch.mean(y, 1, keepdim=True)
        mx  = torch.max(y, 1, keepdim=True)[0]
        att = self.satt(torch.cat([avg, mx], 1))
        return y * att

class FM(nn.Module):
    def __init__(self, c1, c2, k=5, act=True):
        super().__init__()
        c_ = c2 // 2
        self.cv1 = nn.Conv2d(c1, c_, 1, 1, 0, bias=False)
        self.cv2 = nn.Conv2d(c_*4, c2, 1, 1, 0, bias=False)
        self.bn  = nn.BatchNorm2d(c2)
        self.act = SiLU(inplace=True) if act else nn.Identity()
        self.k   = k
    def forward(self, x):
        x = self.cv1(x)
        y1 = F.max_pool2d(x, self.k, 1, self.k//2)
        y2 = F.max_pool2d(y1, self.k, 1, self.k//2)
        y3 = F.max_pool2d(y2, self.k, 1, self.k//2)
        y  = torch.cat([x, y1, y2, y3], 1)
        y  = self.act(self.bn(self.cv2(y)))
        return y

class KWConv(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, kernels=(3,5,7), groups=1, act=True):
        super().__init__()
        self.branches = nn.ModuleList(
            nn.Conv2d(c1, c2, kk, stride=s, padding=kk//2, groups=groups, bias=False)
            for kk in kernels
        )
        self.bn  = nn.BatchNorm2d(c2)
        self.att = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c2, len(kernels), 1, bias=True))
        self.act = SiLU(inplace=True) if act else nn.Identity()
    def forward(self, x):
        feats = [b(x) for b in self.branches]
        stack = torch.stack(feats, dim=1)              
        pooled = stack.sum(1)                          
        pooled = F.adaptive_avg_pool2d(pooled, 1)      
        logits = self.att(pooled).squeeze(-1).squeeze(-1)  
        w = torch.softmax(logits, dim=1).view(x.size(0), -1, 1, 1, 1)
        y = (stack * w).sum(1)                         
        return self.act(self.bn(y))


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Attribute Attention Mechanism ----
class AAM(nn.Module):
    def __init__(self, c, k=3, s=1):
        super().__init__()
        self.conv = nn.Conv2d(c, c, k, s, k//2, bias=False)
        self.bn = nn.BatchNorm2d(c)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

# ---- Efficient Bidirectional Cross Attention ----
class EBCA(nn.Module):
    def __init__(self, c, k=3, s=1):
        super().__init__()
        self.q = nn.Conv2d(c, c, 1, 1)
        self.k = nn.Conv2d(c, c, 1, 1)
        self.v = nn.Conv2d(c, c, 1, 1)

    def forward(self, x):
        q, k, v = self.q(x), self.k(x), self.v(x)
        attn = torch.softmax(torch.matmul(q.flatten(2).transpose(1,2), k.flatten(2)), dim=-1)
        out = torch.matmul(attn, v.flatten(2).transpose(1,2))
        return out.transpose(1,2).reshape_as(x)

# ---- Focal Modulation ----
class FM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Conv2d(c, c, 1)

    def forward(self, x):
        return self.conv(x)

# ---- Kernel Warehouse Convolution ----
class KWConv(nn.Module):
    def __init__(self, c, k_list=[3,7,11]):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv2d(c, c, k, padding=k//2, groups=1) for k in k_list
        ])
        self.bn = nn.BatchNorm2d(c)
        self.act = nn.SiLU()

    def forward(self, x):
        out = sum(conv(x) for conv in self.convs) / len(self.convs)
        return self.act(self.bn(out))


In [ ]:
# %% 1) Paths & config
from pathlib import Path
import torch

# Root dataset directory (adjust to your location)
ROOT = Path(r"d:\omg2\datasets\APPLE")

# YOLO data.yaml path
DATA_YAML = ROOT / "data.yaml"

# Runs folder for training outputs
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

# Frames folder (input for detection later)
FRAMES = ROOT / "frames"

# Detection outputs
DET_OUT = ROOT / "detections_apple"
DET_OUT.mkdir(parents=True, exist_ok=True)

# Video output
DET_VIDEO = ROOT / "detections_apple.mp4"

# Device (GPU if available)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("[cfg] ROOT =", ROOT)
print("[cfg] DATA_YAML =", DATA_YAML)
print("[cfg] RUNS =", RUNS)
print("[cfg] DEVICE =", DEVICE)


[cfg] ROOT = d:\omg2\datasets\APPLE
[cfg] DATA_YAML = d:\omg2\datasets\APPLE\data.yaml
[cfg] RUNS = d:\omg2\datasets\APPLE\runs
[cfg] DEVICE = 0


In [ ]:
# %% Inspect current data.yaml
from pathlib import Path

DATA_YAML = Path(r"d:\omg2\datasets\APPLE\data.yaml")
print("[path]", DATA_YAML)
print(DATA_YAML.read_text())


[path] d:\omg2\datasets\APPLE\data.yaml
path: d:/omg2/datasets/APPLE
train: d:/omg2/datasets/APPLE/images/train
val: d:/omg2/datasets/APPLE/images/val
test: d:/omg2/datasets/APPLE/images/test
names:
  0: apple
nc: 1



In [ ]:
# %% Rewrite data.yaml correctly (train/val/test must point to *images* folders)
from pathlib import Path

ROOT   = Path(r"d:\omg2\datasets\APPLE")
IMAGES = ROOT / "images"
LABELS = ROOT / "labels"
DATA_YAML = ROOT / "data.yaml"

yaml_txt = f"""# YOLO dataset
path: {str(ROOT).replace('\\','/')}
train: {str(IMAGES/'train').replace('\\','/')}
val:   {str(IMAGES/'val').replace('\\','/')}
test:  {str(IMAGES/'test').replace('\\','/')}
nc: 1
names: [apple]
"""
DATA_YAML.write_text(yaml_txt)
print("[wrote] data.yaml ->\n", DATA_YAML.read_text())


[wrote] data.yaml ->
 # YOLO dataset
path: d:/omg2/datasets/APPLE
train: d:/omg2/datasets/APPLE/images/train
val:   d:/omg2/datasets/APPLE/images/val
test:  d:/omg2/datasets/APPLE/images/test
nc: 1
names: [apple]



In [ ]:
# %% Check splits: how many images, missing label files, empty label files, total boxes
from pathlib import Path
import os, re

ROOT   = Path(r"d:\omg2\datasets\APPLE")
IMAGES = ROOT / "images"
LABELS = ROOT / "labels"

def check_split(split):
    import glob
    img_dir = IMAGES / split
    lbl_dir = LABELS / split
    imgs = sorted(glob.glob(str(img_dir/"*.jpg")) + glob.glob(str(img_dir/"*.png")))
    n_img = len(imgs)
    missing = 0
    empty   = 0
    boxes   = 0
    for ip in imgs:
        stem = Path(ip).stem
        lp = lbl_dir / f"{stem}.txt"
        if not lp.exists():
            missing += 1
            continue
        txt = lp.read_text().strip()
        if not txt:
            empty += 1
            continue
        boxes += len([ln for ln in txt.splitlines() if ln.strip()])
    return n_img, missing, empty, boxes

for sp in ["train","val","test"]:
    n_img, miss, emp, box = check_split(sp)
    print(f"[{sp}] images={n_img} | missing_label_files={miss} | empty_label_files={emp} | total_boxes={box}")

# show a sample label file if available
sample = next((LABELS/'train').glob("*.txt"), None)
if sample:
    print("\n[sample label file]", sample)
    print(sample.read_text().splitlines()[:5])
else:
    print("\n[warn] No label files found in labels/train")


[train] images=6160 | missing_label_files=0 | empty_label_files=6160 | total_boxes=0
[val] images=880 | missing_label_files=0 | empty_label_files=880 | total_boxes=0
[test] images=1760 | missing_label_files=0 | empty_label_files=1760 | total_boxes=0

[sample label file] d:\omg2\datasets\APPLE\labels\train\000000.txt
[]


In [ ]:
# %% Delete old .cache files so YOLO rescans labels
from pathlib import Path

for p in [
    Path(r"d:\omg2\datasets\APPLE\labels\train.cache"),
    Path(r"d:\omg2\datasets\APPLE\labels\val.cache"),
    Path(r"d:\omg2\datasets\APPLE\labels\test.cache"),
]:
    if p.exists():
        p.unlink()
        print("[deleted cache]", p)
print("[ok] caches cleared")


[deleted cache] d:\omg2\datasets\APPLE\labels\train.cache
[deleted cache] d:\omg2\datasets\APPLE\labels\val.cache
[ok] caches cleared


In [ ]:
# %% Define enhancement blocks + KWConv and patch_yolo_backbone()

import torch
import torch.nn as nn
from ultralytics import YOLO

# --- Blocks ---
class AAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(4, channels // reduction)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid(),
        )
        self.scale = nn.Parameter(torch.ones(1, channels, 1, 1))

    def forward(self, x):
        w  = self.mlp(self.avg(x))  # channel attn
        xc = x * w
        xs = torch.cat([x.max(1, keepdim=True).values, x.mean(1, keepdim=True)], dim=1)
        ws = self.spatial(xs)       # spatial attn
        xs = x * ws
        xg = x * self.scale         # learnable global scale
        return (xc + xs + xg) / 3.0

class EBCA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw  = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw  = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        y = self.dw(x); y = self.pw(y)
        return self.act(x + y)

class FocalModulation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.local = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.gate  = nn.Sequential(nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid())
    def forward(self, x):
        ctx = self.local(x); g = self.gate(ctx)
        return x * (1 + g)

class KWConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.k3   = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.k5   = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.k7   = nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.fuse = nn.Conv2d(channels * 3, channels, 1, bias=False)
        self.act  = nn.SiLU(inplace=True)
    def forward(self, x):
        y = torch.cat([self.k3(x), self.k5(x), self.k7(x)], dim=1)
        y = self.fuse(y)
        return self.act(x + y)

# --- helper to safely infer out_channels ---
def _infer_out_channels(layer, partial_forward):
    ch = None
    for attr in ("out_channels",):
        if hasattr(layer, attr) and getattr(layer, attr) is not None:
            ch = int(getattr(layer, attr)); break
    if ch is None:
        if hasattr(layer, "cv2") and hasattr(layer.cv2, "out_channels"):
            ch = int(layer.cv2.out_channels)
        elif hasattr(layer, "conv") and hasattr(layer.conv, "out_channels"):
            ch = int(layer.conv.out_channels)
    if ch is not None:
        return ch
    # fallback: run partial forward
    with torch.no_grad():
        feats = partial_forward()
        if isinstance(feats, (list, tuple)):
            feats = feats[-1]
        return int(feats.shape[1]) if hasattr(feats, "shape") else 64

# --- patcher ---
def patch_yolo_backbone(model: YOLO, layers_to_patch=(4,6,8), use_kwconv=True, debug=False):
    m = getattr(model.model, "model", model.model)  # sequential container
    for idx in layers_to_patch:
        if idx >= len(m):
            if debug: print(f"[patch] skip idx={idx} (len={len(m)})")
            continue
        def partial_forward():
            dummy = torch.zeros(1, 3, 64, 64)
            return m[:idx+1](dummy)
        ch = _infer_out_channels(m[idx], partial_forward)
        enh = [AAM(ch), EBCA(ch), FocalModulation(ch)]
        if use_kwconv:
            enh.append(KWConv(ch))
        m[idx] = nn.Sequential(m[idx], nn.Sequential(*enh))
        if debug: print(f"[patch] layer {idx}: ch={ch}, KWConv={use_kwconv}")
    return model

print("[ok] patch_yolo_backbone with AAM+EBCA+FM+KWConv is ready.")


[ok] patch_yolo_backbone with AAM+EBCA+FM+KWConv is ready.


In [ ]:
# %% Sanity check: data.yaml + counts per split
from pathlib import Path

ROOT      = Path(r"d:\omg2\datasets\APPLE")   # <- keep as you used
DATA_YAML = ROOT / "data.yaml"
IM = ROOT / "images"
LB = ROOT / "labels"

assert DATA_YAML.exists(), f"Missing {DATA_YAML}"
for split in ("train","val","test"):
    ims = sorted(list((IM/split).glob("*.jpg")) + list((IM/split).glob("*.png")))
    lbs = sorted(list((LB/split).glob("*.txt")))
    nonempty = [p for p in lbs if p.read_text().strip() != ""]
    print(f"[check] {split:<5} images={len(ims):>5} | labels={len(lbs):>5} | nonempty={len(nonempty):>5}")


[check] train images= 6160 | labels= 6160 | nonempty=    0
[check] val   images=  880 | labels=  880 | nonempty=    0
[check] test  images= 1760 | labels= 1760 | nonempty=    0


In [ ]:
# %% Pseudo-label: create labels/{train,val,test} from a detector
from ultralytics import YOLO
from pathlib import Path

ROOT = Path(r"d:\omg2\datasets\APPLE")
IM   = ROOT / "images"
LB   = ROOT / "labels"
LB.mkdir(exist_ok=True)

# 1) Choose a detector:
#    A) Best from prior run (recommended)
BEST_PT = ROOT / "runs" / "deepHarvestNet-aam-ebca-fm" / "weights" / "best.pt"
#    B) If (A) doesn't exist, fallback to COCO yolov8n.pt
USE_COCO_FALLBACK = not BEST_PT.exists()
MODEL = YOLO("yolov8n.pt" if USE_COCO_FALLBACK else str(BEST_PT))
print("[autolabel] using:", "COCO yolov8n.pt" if USE_COCO_FALLBACK else BEST_PT)

# 2) Resolve which class id to keep (apple or fallback)
names = MODEL.names if hasattr(MODEL, "names") else {}
apple_id = None
for i, n in names.items():
    if str(n).lower().strip() == "apple":
        apple_id = i; break

fallback_order = ["sports ball","orange","banana"]
fallback_id = None; fallback_name = None
if apple_id is None:
    for pname in fallback_order:
        for i, n in names.items():
            if str(n).lower().strip() == pname:
                fallback_id, fallback_name = i, pname; break
        if fallback_id is not None:
            break

keep_id = apple_id if apple_id is not None else fallback_id
print(f"[autolabel] keeping class id: {keep_id} ({'apple' if apple_id is not None else fallback_name})")

def autolabel_split(split, conf=0.20, imgsz=640):
    img_dir = IM / split
    lb_dir  = LB / split
    lb_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
    assert imgs, f"No images in {img_dir}"
    empty = 0
    for i, ip in enumerate(imgs, 1):
        res = MODEL.predict(str(ip), device=0, conf=conf, imgsz=imgsz, verbose=False)[0]
        lines = []
        if keep_id is not None and getattr(res, "boxes", None) is not None and len(res.boxes) > 0:
            cls  = res.boxes.cls.detach().cpu().numpy().astype(int)
            xywh = res.boxes.xywhn.detach().cpu().numpy()
            for c,(x,y,w,h) in zip(cls,xywh):
                if c == keep_id:
                    lines.append(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
        (lb_dir / f"{ip.stem}.txt").write_text("\n".join(lines))
        if not lines: empty += 1
        if i == 1 or i % 200 == 0 or i == len(imgs):
            print(f"[autolabel {split}] {i}/{len(imgs)}")
    print(f"[autolabel {split}] empty={empty} / {len(imgs)}")

for split in ("train","val","test"):
    autolabel_split(split, conf=0.20, imgsz=640)

print("[autolabel] done ->", LB)


[autolabel] using: COCO yolov8n.pt
[autolabel] keeping class id: 47 (apple)
[autolabel train] 1/6160
[autolabel train] 200/6160
[autolabel train] 400/6160
[autolabel train] 600/6160
[autolabel train] 800/6160
[autolabel train] 1000/6160
[autolabel train] 1200/6160
[autolabel train] 1400/6160
[autolabel train] 1600/6160
[autolabel train] 1800/6160
[autolabel train] 2000/6160
[autolabel train] 2200/6160
[autolabel train] 2400/6160
[autolabel train] 2600/6160
[autolabel train] 2800/6160
[autolabel train] 3000/6160
[autolabel train] 3200/6160
[autolabel train] 3400/6160
[autolabel train] 3600/6160
[autolabel train] 3800/6160
[autolabel train] 4000/6160
[autolabel train] 4200/6160
[autolabel train] 4400/6160
[autolabel train] 4600/6160
[autolabel train] 4800/6160
[autolabel train] 5000/6160
[autolabel train] 5200/6160
[autolabel train] 5400/6160
[autolabel train] 5600/6160
[autolabel train] 5800/6160
[autolabel train] 6000/6160
[autolabel train] 6160/6160
[autolabel train] empty=108 / 6160


In [ ]:
# %% Train with KWConv (dataset fixed)
from ultralytics import YOLO
from pathlib import Path

EPOCHS, IMGSZ, BATCH, WORKERS, AMP = 100, 640, 8, 2, True
ROOT  = Path(r"d:\omg2\datasets\APPLE")
RUNS  = ROOT / "runs"
DATA_YAML = ROOT / "data.yaml"
DEVICE = 0

print("[train] starting with KWConv enabled (dataset fixed)...")
model = YOLO("yolov8n.pt")
model = patch_yolo_backbone(model, layers_to_patch=(4,6,8), use_kwconv=True, debug=True)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    amp=AMP,
    project=str(RUNS),
    name="deepHarvestNet-aam-ebca-fm-kwconv-fixed",
    verbose=True,
    cos_lr=True,
)
print("[train] done ->", results.save_dir)


[train] starting with KWConv enabled (dataset fixed)...
[patch] layer 4: ch=64, KWConv=True
[patch] layer 6: ch=128, KWConv=True
[patch] layer 8: ch=256, KWConv=True
New https://pypi.org/project/ultralytics/8.3.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.198  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\omg2\datasets\APPLE\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, l

In [ ]:
# %% Detect apples on frames -> annotated images + MP4
from pathlib import Path
import glob, cv2, os
from ultralytics import YOLO

# ---------------------------------------------------------
# 0) Expect these from your config; set safe fallbacks if missing
# ---------------------------------------------------------
try:
    ROOT
except NameError:
    ROOT = Path(r"d:\omg2\datasets\APPLE")
try:
    RUNS
except NameError:
    RUNS = ROOT / "runs"
try:
    DEVICE
except NameError:
    DEVICE = 0
try:
    IMGSZ
except NameError:
    IMGSZ = 640

ROOT  = Path(ROOT)
RUNS  = Path(RUNS)

# ---------------------------------------------------------
# 1) Find trained weights (prefers your KWConv runs)
# ---------------------------------------------------------
preferred_runs = [
    "deepHarvestNet-aam-ebca-fm-kwconv-fixed",
    "deepHarvestNet-aam-ebca-fm-kwconv3",
    "deepHarvestNet-aam-ebca-fm-kwconv32",
    "deepHarvestNet-aam-ebca-fm-kwconv2",
    "deepHarvestNet-aam-ebca-fm-kwconv",
    "deepHarvestNet-aam-ebca-fm",
]

BEST_PT = None
for rn in preferred_runs:
    p = RUNS / rn / "weights" / "best.pt"
    if p.exists():
        BEST_PT = p
        break

if BEST_PT is None:
    # fallback: pick the newest/longest match under runs/*/weights/best.pt
    candidates = sorted(glob.glob(str(RUNS / "*" / "weights" / "best.pt")))
    if candidates:
        BEST_PT = Path(candidates[-1])

assert BEST_PT is not None and BEST_PT.exists(), f"Trained weights not found under {RUNS}. No best.pt available."

print(f"[infer] using weights -> {BEST_PT}")

# ---------------------------------------------------------
# 2) Choose frames folder
#    Adjust order below to your layout (it matches your screenshot)
# ---------------------------------------------------------
candidates = [
    ROOT / "frames_800",
    ROOT / "frames_80fps_exact",
    ROOT / "frames_80fps",
    ROOT / "frames",
    ROOT / "images" / "test",   # YOLO-style split
    ROOT / "images",            # generic
]

FRAMES = None
for c in candidates:
    if c.exists() and any(c.glob("*.jpg")) or any(c.glob("*.png")):
        FRAMES = c
        break

assert FRAMES is not None and FRAMES.exists(), "Could not find a frames folder with images. Update 'candidates'."
print(f"[infer] frames dir -> {FRAMES}")

# ---------------------------------------------------------
# 3) Output folders
# ---------------------------------------------------------
DET_OUT   = ROOT / "detections_apple"
DET_OUT.mkdir(parents=True, exist_ok=True)
DET_VIDEO = ROOT / "apple_detections.mp4"

# Clear previous annotated images (keep video)
for f in DET_OUT.glob("*"):
    try:
        f.unlink()
    except Exception:
        pass

# ---------------------------------------------------------
# 4) Run detection and save annotated frames
# ---------------------------------------------------------
model = YOLO(str(BEST_PT))
img_paths = sorted(list(FRAMES.glob("*.jpg")) + list(FRAMES.glob("*.png")))
assert img_paths, f"No images found in {FRAMES}"

for i, p in enumerate(img_paths, 1):
    res = model.predict(
        source=str(p),
        device=DEVICE,
        conf=0.25,
        imgsz=IMGSZ,
        iou=0.5,
        verbose=False
    )[0]
    annotated = res.plot()  # BGR ndarray
    out_path  = DET_OUT / p.name
    cv2.imwrite(str(out_path), annotated)

    if i == 1 or i % 200 == 0 or i == len(img_paths):
        print(f"[detect] {i}/{len(img_paths)} -> {out_path}")

print(f"[detect] done. Annotated frames -> {DET_OUT}")

# ---------------------------------------------------------
# 5) Stitch annotated frames into MP4
#    FPS: try to inherit from a known raw video near frames, else default 30.
# ---------------------------------------------------------
def _guess_fps():
    # Try to find a sibling/nearby video to borrow fps from
    guesses = list(ROOT.glob("*.mp4")) + list(ROOT.glob("*.mov")) + list(ROOT.glob("*.avi"))
    for vid in guesses:
        cap = cv2.VideoCapture(str(vid))
        if cap.isOpened():
            f = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
            if f and f > 0:
                return float(f)
    return 30.0

fps = _guess_fps()

# Infer frame size from first annotated image
first = next(iter(sorted(DET_OUT.glob("*.jpg"))), None)
if first is None:
    first = next(iter(sorted(DET_OUT.glob("*.png"))), None)
assert first is not None, f"No annotated frames in {DET_OUT}."

img0 = cv2.imread(str(first))
h, w = img0.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter(str(DET_VIDEO), fourcc, fps, (w, h))
assert vw.isOpened(), f"Could not open writer for {DET_VIDEO}"

frames_sorted = sorted(list(DET_OUT.glob("*.jpg")) + list(DET_OUT.glob("*.png")))
for i, f in enumerate(frames_sorted, 1):
    im = cv2.imread(str(f))
    if im is None:
        continue
    if im.shape[0] != h or im.shape[1] != w:
        im = cv2.resize(im, (w, h), interpolation=cv2.INTER_LINEAR)
    vw.write(im)
    if i == 1 or i % 200 == 0 or i == len(frames_sorted):
        print(f"[video] {i}/{len(frames_sorted)}")

vw.release()
print(f"[video] saved -> {DET_VIDEO}")


[infer] using weights -> d:\omg2\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv-fixed\weights\best.pt
[infer] frames dir -> d:\omg2\datasets\APPLE\frames_800
[detect] 1/800 -> d:\omg2\datasets\APPLE\detections_apple\000000.jpg
[detect] 200/800 -> d:\omg2\datasets\APPLE\detections_apple\001138.jpg
[detect] 400/800 -> d:\omg2\datasets\APPLE\detections_apple\002283.jpg
[detect] 600/800 -> d:\omg2\datasets\APPLE\detections_apple\003427.jpg
[detect] 800/800 -> d:\omg2\datasets\APPLE\detections_apple\004572.jpg
[detect] done. Annotated frames -> d:\omg2\datasets\APPLE\detections_apple
[video] 1/800
[video] 200/800
[video] 400/800
[video] 600/800
[video] 800/800
[video] saved -> d:\omg2\datasets\APPLE\apple_detections.mp4


In [ ]:
# %% Show a couple of metrics if available
try:
    from matplotlib import pyplot as plt
    hist = results.metrics  # ultralytics object with logged arrays
    # Some versions use results.results_dict or results.metrics dicts; guard accordingly:
    if hasattr(results, "results_dict"):
        print("[results]", results.results_dict)

    # If arrays are available, plot them
    if isinstance(hist, dict):
        if "train/box_loss" in hist:
            plt.figure()
            plt.plot(hist["train/box_loss"], label="box_loss")
            plt.plot(hist["train/cls_loss"], label="cls_loss")
            plt.plot(hist["train/dfl_loss"], label="dfl_loss")
            plt.legend(); plt.title("Training losses"); plt.xlabel("epoch"); plt.show()
        if "metrics/mAP_0.5" in hist:
            plt.figure()
            plt.plot(hist["metrics/mAP_0.5"], label="mAP@0.5")
            if "metrics/mAP_0.5:0.95" in hist:
                plt.plot(hist["metrics/mAP_0.5:0.95"], label="mAP@0.5:0.95")
            plt.legend(); plt.title("Validation mAP"); plt.xlabel("epoch"); plt.show()
except Exception as e:
    print("[plot] skipped:", e)


[plot] skipped: 'DetMetrics' object has no attribute 'metrics'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        names (dict[int, str]): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
        task (str): The task type, set to 'detect'.
        stats (dict[str, list]): A dictionary containing lists for true positives, confidence scores, predicted classes, target classes, and target images.
        nt_per_class: Number of targets per class.
        nt_per_image: Number of targets per image.

    Methods:
        update_stats: Update statistics by appending new values to existing stat collections.
        process: Process predicted results for object detection and update metrics.
        cle

In [ ]:
# %% 2) AAM, EBCA, FocalModulation, KWConv + robust YOLO patcher
import torch
import torch.nn as nn
from ultralytics import YOLO

class AAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(4, channels // reduction)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid(),
        )
        self.scale = nn.Parameter(torch.ones(1, channels, 1, 1))

    def forward(self, x):
        w  = self.mlp(self.avg(x))                 # channel
        xc = x * w
        xs = torch.cat([x.max(1, keepdim=True).values,
                        x.mean(1, keepdim=True)], dim=1)
        ws = self.spatial(xs)                      # spatial
        xs = x * ws
        xg = x * self.scale                        # global learnable
        return (xc + xs + xg) / 3.0

class EBCA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw  = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw  = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        y = self.dw(x); y = self.pw(y)
        return self.act(x + y)

class FocalModulation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.local = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.gate  = nn.Sequential(nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid())
    def forward(self, x):
        ctx = self.local(x); g = self.gate(ctx)
        return x * (1 + g)

class KWConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.k3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.k5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.k7 = nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.fuse = nn.Conv2d(channels*3, channels, 1, bias=False)
        self.act  = nn.SiLU(inplace=True)
    def forward(self, x):
        a = self.k3(x); b = self.k5(x); c = self.k7(x)
        y = torch.cat([a,b,c], dim=1); y = self.fuse(y)
        return self.act(x + y)

# ---- patch helpers ----
def _infer_out_channels(layer, fallback_run):
    if hasattr(layer, "out_channels") and layer.out_channels is not None:
        return int(layer.out_channels)
    if hasattr(layer, "cv2") and hasattr(layer.cv2, "out_channels"):
        return int(layer.cv2.out_channels)
    if hasattr(layer, "conv") and hasattr(layer.conv, "out_channels"):
        return int(layer.conv.out_channels)
    try:
        with torch.no_grad():
            feats = fallback_run()
            if isinstance(feats, (list, tuple)): feats = feats[-1]
            return int(feats.shape[1])
    except Exception:
        return 64

def patch_yolo_backbone(model: YOLO, layers_to_patch=(4,6,8), use_kwconv=True, debug=False):
    m = getattr(model.model, "model", None) or model.model
    for idx in layers_to_patch:
        if idx >= len(m): 
            if debug: print(f"[patch] skip idx={idx} (len={len(m)})")
            continue
        def _fallback_run():
            dummy = torch.zeros(1,3,64,64)
            return m[:idx+1](dummy)
        ch = _infer_out_channels(m[idx], _fallback_run)
        layers = [AAM(ch), EBCA(ch), FocalModulation(ch)]
        if use_kwconv: layers.append(KWConv(ch))
        block = nn.Sequential(*layers)
        m[idx] = nn.Sequential(m[idx], block)
        if debug: print(f"[patch] layer {idx}: ch={ch}, KWConv={use_kwconv}")
    return model

print("[modules] AAM, EBCA, FM, KWConv + patcher ready.")


[modules] AAM, EBCA, FM, KWConv + patcher ready.


In [ ]:
# %% 3) AdaBins-style depth head that fuses YOLO neck features (P3,P4,P5)
# This is a lightweight, AdaBins-inspired head: upsample P4/P5 to P3, fuse, predict 1-channel depth.
import torch.nn.functional as F

class DepthHeadAdaBinsLite(nn.Module):
    """
    Fuses multi-scale features (P3,P4,P5): upsample to P3 size -> concat -> conv tower -> 1-ch depth.
    Use Softplus to keep depth positive (you can rescale later).
    """
    def __init__(self, in_c_p3=128, in_c_p4=256, in_c_p5=256, mid=128):
        super().__init__()
        fuse_c = in_c_p3 + in_c_p4 + in_c_p5
        self.reduce = nn.Sequential(
            nn.Conv2d(fuse_c, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
        )
        self.block = nn.Sequential(
            nn.Conv2d(mid, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.SiLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.SiLU(inplace=True),
        )
        self.pred = nn.Conv2d(mid, 1, 1)        # raw depth logits
        self.softplus = nn.Softplus(beta=1.0)   # positive depth

    def forward(self, p3, p4, p5):
        H, W = p3.shape[-2:]
        p4u = F.interpolate(p4, size=(H,W), mode="nearest")
        p5u = F.interpolate(p5, size=(H,W), mode="nearest")
        x   = torch.cat([p3, p4u, p5u], dim=1)
        x   = self.reduce(x)
        x   = self.block(x)
        d   = self.softplus(self.pred(x))       # [B,1,H,W], positive
        return d

print("[depth] AdaBins-lite head ready.")


[depth] AdaBins-lite head ready.


In [ ]:
# %% 4) Hook YOLO to grab neck feature maps (P3,P4,P5) and run depth head
# Ultralytics v8 Detect graph: after SPPF (9) -> top-down upsample/concat -> C2f -> ... -> three scales.
# We'll hook at indices that correspond to neck C2f outputs feeding Detect:
#    idx 15 ~ P3 (small), idx 18 ~ P4 (medium), idx 21 ~ P5 (large) for v8n/standard models.
# If you changed architecture heavily, adjust indices accordingly (print summary and verify).

from types import SimpleNamespace

class DepthFromYOLO(nn.Module):
    def __init__(self, yolo_model: YOLO, p3_idx=15, p4_idx=18, p5_idx=21,
                 ch_p3=64, ch_p4=128, ch_p5=256):
        super().__init__()
        self.yolo = yolo_model
        self.model_list = getattr(self.yolo.model, "model", None) or self.yolo.model
        self.p3_idx, self.p4_idx, self.p5_idx = p3_idx, p4_idx, p5_idx

        self.depth_head = DepthHeadAdaBinsLite(in_c_p3=ch_p3, in_c_p4=ch_p4, in_c_p5=ch_p5, mid=128)
        self._hooks = []
        self._buffers = SimpleNamespace(p3=None, p4=None, p5=None)

        # register forward hooks
        def make_hook(which):
            def _hook(_, __, out):
                setattr(self._buffers, which, out)
            return _hook

        self._hooks.append(self.model_list[self.p3_idx].register_forward_hook(make_hook("p3")))
        self._hooks.append(self.model_list[self.p4_idx].register_forward_hook(make_hook("p4")))
        self._hooks.append(self.model_list[self.p5_idx].register_forward_hook(make_hook("p5")))

    @torch.no_grad()
    def infer(self, img_bgr, device=0, imgsz=640, yolo_conf=0.25, yolo_iou=0.5):
        """
        img_bgr: numpy BGR image (H,W,3)
        returns: yolo_results (ultralytics Results), depth_map (torch tensor [1,1,H2,W2] at P3 scale)
        """
        # run YOLO predict (this will fill p3/p4/p5 buffers via hooks)
        res = self.yolo.predict(img_bgr, imgsz=imgsz, device=device,
                                conf=yolo_conf, iou=yolo_iou, verbose=False)[0]
        # fetch features
        p3, p4, p5 = self._buffers.p3, self._buffers.p4, self._buffers.p5
        assert (p3 is not None) and (p4 is not None) and (p5 is not None), "Neck hooks didn't trigger—check indices."

        # compute depth at P3 scale
        depth = self.depth_head(p3, p4, p5)     # [1,1,h,w]
        return res, depth

    def close(self):
        for h in self._hooks: h.remove()
        self._hooks = []

print("[depth] YOLO feature hook wrapper ready.")


[depth] YOLO feature hook wrapper ready.


In [ ]:
# %% Fixed DepthFromYOLO (do NOT overwrite Module._buffers)
import torch
import torch.nn as nn
from types import SimpleNamespace

class DepthFromYOLO(nn.Module):
    """
    Hook P3/P4/P5 feature maps from an Ultralytics YOLO model and predict depth.
    Assumes P3,P4,P5 correspond to layers 15,18,21 on YOLOv8n (check your summary).
    """
    def __init__(self, yolo_model, p3_idx=15, p4_idx=18, p5_idx=21,
                 ch_p3=64, ch_p4=128, ch_p5=256):
        super().__init__()
        self.det = yolo_model
        # Ultralytics stores the sequential blocks at model.model (list-like)
        self.model_list = getattr(self.det.model, "model", self.det.model)

        self.p3_idx, self.p4_idx, self.p5_idx = p3_idx, p4_idx, p5_idx

        # ➜ Use our own namespace; DON'T call it _buffers.
        self._feats = SimpleNamespace(p3=None, p4=None, p5=None)
        self._hooks = []

        def _make_hook(which):
            def _hook(_, __, out):
                # out can be a list/tuple; grab last if so
                if isinstance(out, (list, tuple)):
                    out = out[-1]
                setattr(self._feats, which, out)
            return _hook

        # Register hooks on the selected layers
        self._hooks.append(self.model_list[self.p3_idx].register_forward_hook(_make_hook("p3")))
        self._hooks.append(self.model_list[self.p4_idx].register_forward_hook(_make_hook("p4")))
        self._hooks.append(self.model_list[self.p5_idx].register_forward_hook(_make_hook("p5")))

        # Simple AdaBins-style coarse-to-fine depth head (toy version)
        def head(cin):
            return nn.Sequential(
                nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False),
                nn.Conv2d(cin, cin//2, 1, bias=False),
                nn.SiLU(inplace=True),
            )

        self.h3 = head(ch_p3)
        self.h4 = head(ch_p4)
        self.h5 = head(ch_p5)

        # Fuse P3,P4,P5 (after upsampling)
        self.fuse = nn.Sequential(
            nn.Conv2d(ch_p3//2 + ch_p4//2 + ch_p5//2, 128, 3, padding=1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(128, 1, 1)  # 1-channel depth map (relative)
        )

    def forward(self, x):
        """
        x: BCHW image tensor. We forward through the detector once to populate hooks,
        then read P3/P4/P5 from self._feats and produce a depth map.
        """
        _ = self.det.model(x)  # forward through the backbone/neck/head (hooks capture features)

        p3 = self._feats.p3
        p4 = self._feats.p4
        p5 = self._feats.p5
        assert p3 is not None and p4 is not None and p5 is not None, "Hooks did not capture features. Check layer indices."

        f3 = self.h3(p3)
        f4 = nn.functional.interpolate(self.h4(p4), size=f3.shape[-2:], mode="bilinear", align_corners=False)
        f5 = nn.functional.interpolate(self.h5(p5), size=f3.shape[-2:], mode="bilinear", align_corners=False)

        fused = torch.cat([f3, f4, f5], dim=1)
        depth = self.fuse(fused)
        return depth  # Bx1xHxW

    def remove_hooks(self):
        for h in self._hooks:
            try: h.remove()
            except: pass
        self._hooks = []


In [ ]:
# %% Build wrapper with your trained model and verify feature indices
from ultralytics import YOLO
import torch
from pathlib import Path

# Expect BEST_PT, RUNS, DEVICE to be set already; otherwise set them:
try:
    BEST_PT
except NameError:
    # pick a likely run
    BEST_PT = Path(r"d:\omg2\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv-fixed\weights\best.pt")
detector = YOLO(str(BEST_PT))

# If you changed the YOLO graph, print to confirm indices:
# print(detector.model)  # uncomment to inspect and confirm 15,18,21 are the P3,P4,P5 C2f blocks

depth_wrapper = DepthFromYOLO(
    detector,
    p3_idx=15, p4_idx=18, p5_idx=21,  # adjust if your printed model differs
    ch_p3=64, ch_p4=128, ch_p5=256    # v8n defaults after neck
).to(DEVICE if isinstance(DEVICE, int) else 0)

# Quick forward to confirm shapes
dummy = torch.zeros(1, 3, 640, 640).to(next(detector.model.parameters()).device)
with torch.no_grad():
    depth_map = depth_wrapper(dummy)
print("[check] depth shape:", tuple(depth_map.shape))  # expect (1,1,h,w) roughly at P3 resolution (e.g., 80x80)


[check] depth shape: (1, 1, 80, 80)


In [ ]:
# %% 1) Paths & model (safe defaults)
from pathlib import Path
import torch
from ultralytics import YOLO

# ---- config (edit if needed) ----
ROOT   = Path(r"d:\omg2\datasets\APPLE")
RUNS   = ROOT / "runs"
FRAMES = ROOT / "frames_800"           # <--- change to your frames folder if different
IMGSZ  = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Pick trained weights; fallback to last.pt or base if missing
cands = [
    RUNS / "deepHarvestNet-aam-ebca-fm-kwconv-fixed" / "weights" / "best.pt",
    RUNS / "deepHarvestNet-aam-ebca-fm-kwconv-fixed" / "weights" / "last.pt",
    RUNS / "deepHarvestNet-aam-ebca-fm-kwconv" / "weights" / "best.pt",
    RUNS / "deepHarvestNet-aam-ebca-fm-kwconv" / "weights" / "last.pt",
    Path("yolov8n.pt")
]
BEST_PT = next((p for p in cands if p.exists()), cands[-1])
print("[cfg] FRAMES =", FRAMES)
print("[cfg] BEST_PT =", BEST_PT)

# Load detector
detector = YOLO(str(BEST_PT))

# Build your depth wrapper (P3,P4,P5 indices match v8n; adjust if you changed graph)
depth_wrapper = DepthFromYOLO(detector, p3_idx=15, p4_idx=18, p5_idx=21,
                              ch_p3=64, ch_p4=128, ch_p5=256).to(DEVICE)
print("[load] detector + depth wrapper ready")


[cfg] FRAMES = d:\omg2\datasets\APPLE\frames_800
[cfg] BEST_PT = d:\omg2\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv-fixed\weights\best.pt
[load] detector + depth wrapper ready


In [ ]:
# %% 2) Detect apples on frames -> annotated images
import cv2

assert FRAMES.exists(), f"Frames folder not found: {FRAMES}"
DET_DIR = ROOT / "detections_apple"
DET_DIR.mkdir(parents=True, exist_ok=True)

img_paths = sorted(list(FRAMES.glob("*.jpg")) + list(FRAMES.glob("*.png")))
assert img_paths, f"No images found in {FRAMES}"

# clear old outputs
for f in DET_DIR.glob("*"): 
    try: f.unlink()
    except: pass

for i, p in enumerate(img_paths, 1):
    res = detector.predict(str(p), device=DEVICE, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    annotated = res.plot()  # BGR numpy array with boxes
    out_path = DET_DIR / p.name
    cv2.imwrite(str(out_path), annotated)
    if i == 1 or i % 200 == 0 or i == len(img_paths):
        print(f"[detect] {i}/{len(img_paths)} -> {out_path}")

print(f"[detect] done. Frames saved at: {DET_DIR}")


[detect] 1/800 -> d:\omg2\datasets\APPLE\detections_apple\000000.jpg
[detect] 200/800 -> d:\omg2\datasets\APPLE\detections_apple\001138.jpg
[detect] 400/800 -> d:\omg2\datasets\APPLE\detections_apple\002283.jpg
[detect] 600/800 -> d:\omg2\datasets\APPLE\detections_apple\003427.jpg
[detect] 800/800 -> d:\omg2\datasets\APPLE\detections_apple\004572.jpg
[detect] done. Frames saved at: d:\omg2\datasets\APPLE\detections_apple


In [ ]:
# %% 3) Depth inference -> colorized depth + side-by-side panels
import cv2
import numpy as np

DEPTH_DIR   = ROOT / "depth_maps"
PANEL_DIR   = ROOT / "det_depth_panels"
for d in (DEPTH_DIR, PANEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
    for f in d.glob("*"):
        try: f.unlink()
        except: pass

def colorize_depth(depth_1xHxW, out_hw):
    """depth_1xHxW (numpy float32), min-max normalize, resize, apply colormap."""
    d = depth_1xHxW
    d = d - d.min() if d.size and np.isfinite(d).any() else d
    mx = d.max() if d.size else 1.0
    d = d / (mx + 1e-8)
    d = (d * 255.0).clip(0,255).astype(np.uint8)
    d = cv2.resize(d, out_hw[::-1], interpolation=cv2.INTER_CUBIC)  # (w,h)
    d_color = cv2.applyColorMap(d, cv2.COLORMAP_INFERNO)
    return d_color

for i, p in enumerate(img_paths, 1):
    # read and prepare tensor for depth wrapper
    bgr = cv2.imread(str(p))
    assert bgr is not None, f"Cannot read {p}"
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_res = cv2.resize(rgb, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
    t = torch.from_numpy(rgb_res).to(DEVICE).permute(2,0,1).float().unsqueeze(0) / 255.0

    with torch.no_grad():
        depth = depth_wrapper(t)         # Bx1xhxw
    depth_np = depth.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)
    depth_c  = colorize_depth(depth_np, out_hw=(bgr.shape[0], bgr.shape[1]))  # HxW color

    # load detection render from previous cell
    det_img_path = (ROOT / "detections_apple" / p.name)
    if det_img_path.exists():
        det_img = cv2.imread(str(det_img_path))
    else:
        # safety: render now if missing
        res = detector.predict(str(p), device=DEVICE, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
        det_img = res.plot()

    # side-by-side panel
    panel = np.hstack([det_img, depth_c])
    cv2.imwrite(str(PANEL_DIR / p.name), panel)

    # also save depth map alone
    cv2.imwrite(str(DEPTH_DIR / p.stem) + "_depth.png", depth_c)

    if i == 1 or i % 200 == 0 or i == len(img_paths):
        print(f"[depth] {i}/{len(img_paths)} -> {PANEL_DIR / p.name}")

print(f"[depth] panels saved at: {PANEL_DIR}")


[depth] 1/800 -> d:\omg2\datasets\APPLE\det_depth_panels\000000.jpg
[depth] 200/800 -> d:\omg2\datasets\APPLE\det_depth_panels\001138.jpg
[depth] 400/800 -> d:\omg2\datasets\APPLE\det_depth_panels\002283.jpg
[depth] 600/800 -> d:\omg2\datasets\APPLE\det_depth_panels\003427.jpg
[depth] 800/800 -> d:\omg2\datasets\APPLE\det_depth_panels\004572.jpg
[depth] panels saved at: d:\omg2\datasets\APPLE\det_depth_panels


In [ ]:
# %% 4) Stitch panels into MP4
import cv2

PANEL_VIDEO = ROOT / "apple_detect_depth.mp4"

# fps heuristic: try to adopt from a source video name nearby, else 30
fps = 30.0
try:
    # If you have a RAW video path, set it here to inherit FPS
    RAW_VIDEO = next((v for v in [ROOT/"raw.mp4", ROOT/"raw_video.mp4"] if v.exists()), None)
    if RAW_VIDEO:
        cap_src = cv2.VideoCapture(str(RAW_VIDEO))
        if cap_src.isOpened():
            f = cap_src.get(cv2.CAP_PROP_FPS)
            if f and f > 0: fps = f
        cap_src.release()
except Exception:
    pass

frames_sorted = sorted(list(PANEL_DIR.glob("*.jpg")) + list(PANEL_DIR.glob("*.png")))
assert frames_sorted, f"No panels found in {PANEL_DIR}. Run previous cell first."

# infer size
first = frames_sorted[0]
im0 = cv2.imread(str(first))
h, w = im0.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter(str(PANEL_VIDEO), fourcc, fps, (w, h))
assert vw.isOpened(), f"Could not open writer: {PANEL_VIDEO}"

for i, f in enumerate(frames_sorted, 1):
    img = cv2.imread(str(f))
    if img is None: 
        continue
    if img.shape[:2] != (h, w):
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)
    vw.write(img)
    if i == 1 or i % 200 == 0 or i == len(frames_sorted):
        print(f"[video] {i}/{len(frames_sorted)}")

vw.release()
print(f"[video] saved -> {PANEL_VIDEO}")


[video] 1/800
[video] 200/800
[video] 400/800
[video] 600/800
[video] 800/800
[video] saved -> d:\omg2\datasets\APPLE\apple_detect_depth.mp4


In [ ]:
# %% Detect apples and overlay depth values on bounding boxes
import cv2
import numpy as np
import torch

DET_DEPTH_DIR = ROOT / "detections_with_depth"
DET_DEPTH_DIR.mkdir(parents=True, exist_ok=True)

# clear previous
for f in DET_DEPTH_DIR.glob("*"): 
    try: f.unlink()
    except: pass

for i, p in enumerate(img_paths, 1):
    # original image
    bgr = cv2.imread(str(p))
    h0, w0 = bgr.shape[:2]

    # run detection
    res = detector.predict(str(p), device=DEVICE, conf=0.25, imgsz=IMGSZ, verbose=False)[0]

    # run depth
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_res = cv2.resize(rgb, (IMGSZ, IMGSZ))
    t = torch.from_numpy(rgb_res).to(DEVICE).permute(2,0,1).float().unsqueeze(0) / 255.0
    with torch.no_grad():
        depth = depth_wrapper(t)
    depth_np = depth.squeeze().detach().cpu().numpy()

    # normalize depth to meters (simple heuristic, adjust scaling factor if needed)
    dmin, dmax = depth_np.min(), depth_np.max()
    norm_depth = (depth_np - dmin) / (dmax - dmin + 1e-8)
    depth_meters = 0.5 + norm_depth * 1.5   # scale [0.5m, 2.0m], tweak as needed

    # draw detections with depth
    for box in res.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        conf = float(res.boxes.conf[0].cpu().numpy()) if len(res.boxes.conf) else 0.0

        # sample depth at box center
        cx, cy = (x1+x2)//2, (y1+y2)//2
        dx = int(cx / w0 * depth_np.shape[1])
        dy = int(cy / h0 * depth_np.shape[0])
        if 0 <= dx < depth_np.shape[1] and 0 <= dy < depth_np.shape[0]:
            dist_val = depth_meters[dy, dx]
        else:
            dist_val = np.nan

        label = f"apple {conf:.2f} dist {dist_val:.2f}m"

        cv2.rectangle(bgr, (x1,y1), (x2,y2), (0,0,255), 2)
        cv2.putText(bgr, label, (x1, max(y1-5,15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)

    out_path = DET_DEPTH_DIR / p.name
    cv2.imwrite(str(out_path), bgr)

    if i == 1 or i % 200 == 0 or i == len(img_paths):
        print(f"[detect+depth] {i}/{len(img_paths)} -> {out_path}")

print(f"[detect+depth] done. Saved at {DET_DEPTH_DIR}")


[detect+depth] 1/800 -> d:\omg2\datasets\APPLE\detections_with_depth\000000.jpg
[detect+depth] 200/800 -> d:\omg2\datasets\APPLE\detections_with_depth\001138.jpg
[detect+depth] 400/800 -> d:\omg2\datasets\APPLE\detections_with_depth\002283.jpg
[detect+depth] 600/800 -> d:\omg2\datasets\APPLE\detections_with_depth\003427.jpg
[detect+depth] 800/800 -> d:\omg2\datasets\APPLE\detections_with_depth\004572.jpg
[detect+depth] done. Saved at d:\omg2\datasets\APPLE\detections_with_depth


In [ ]:
# %% Detect apples and overlay depth values (bold + dark labels)
import cv2
import numpy as np
import torch

DET_DEPTH_DIR = ROOT / "detections_with_depth_bold"
DET_DEPTH_DIR.mkdir(parents=True, exist_ok=True)

# clear previous outputs
for f in DET_DEPTH_DIR.glob("*"):
    try: f.unlink()
    except: pass

for i, p in enumerate(img_paths, 1):
    bgr = cv2.imread(str(p))
    h0, w0 = bgr.shape[:2]

    res = detector.predict(str(p), device=DEVICE, conf=0.25, imgsz=IMGSZ, verbose=False)[0]

    # depth estimation
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_res = cv2.resize(rgb, (IMGSZ, IMGSZ))
    t = torch.from_numpy(rgb_res).to(DEVICE).permute(2,0,1).float().unsqueeze(0) / 255.0
    with torch.no_grad():
        depth = depth_wrapper(t)
    depth_np = depth.squeeze().detach().cpu().numpy()
    dmin, dmax = depth_np.min(), depth_np.max()
    norm_depth = (depth_np - dmin) / (dmax - dmin + 1e-8)
    depth_meters = 0.5 + norm_depth * 1.5

    for box, conf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = map(int, box)
        cx, cy = (x1+x2)//2, (y1+y2)//2
        dx = int(cx / w0 * depth_np.shape[1])
        dy = int(cy / h0 * depth_np.shape[0])
        dist_val = depth_meters[dy, dx] if 0 <= dx < depth_np.shape[1] and 0 <= dy < depth_np.shape[0] else np.nan

        label = f"apple {conf:.2f} dist {dist_val:.2f}m"

        # draw box
        cv2.rectangle(bgr, (x1,y1), (x2,y2), (0,0,255), 2)

        # draw text with dark background
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.8   # bigger font
        thickness = 2
        (tw, th), baseline = cv2.getTextSize(label, font, scale, thickness)
        cv2.rectangle(bgr, (x1, y1-th-6), (x1+tw, y1), (0,0,0), -1)  # black background
        cv2.putText(bgr, label, (x1, y1-5), font, scale, (255,255,255), thickness, cv2.LINE_AA)

    out_path = DET_DEPTH_DIR / p.name
    cv2.imwrite(str(out_path), bgr)

    if i == 1 or i % 200 == 0 or i == len(img_paths):
        print(f"[detect+depth-bold] {i}/{len(img_paths)} -> {out_path}")

print(f"[detect+depth-bold] done. Saved at {DET_DEPTH_DIR}")


[detect+depth-bold] 1/800 -> d:\omg2\datasets\APPLE\detections_with_depth_bold\000000.jpg
[detect+depth-bold] 200/800 -> d:\omg2\datasets\APPLE\detections_with_depth_bold\001138.jpg
[detect+depth-bold] 400/800 -> d:\omg2\datasets\APPLE\detections_with_depth_bold\002283.jpg
[detect+depth-bold] 600/800 -> d:\omg2\datasets\APPLE\detections_with_depth_bold\003427.jpg
[detect+depth-bold] 800/800 -> d:\omg2\datasets\APPLE\detections_with_depth_bold\004572.jpg
[detect+depth-bold] done. Saved at d:\omg2\datasets\APPLE\detections_with_depth_bold


In [ ]:
import cv2
from pathlib import Path

# Folder with the images you already wrote (e.g., from your detect+depth-bold step)
SRC_DIR = ROOT / "detections_with_depth_bold"   # change if you used another folder
OUT_MP4 = ROOT / "detections_with_depth_bold_from_frames.mp4"
FPS     = 30.0

imgs = sorted(list(SRC_DIR.glob("*.jpg")) + list(SRC_DIR.glob("*.png")))
assert imgs, f"No images found in {SRC_DIR}"

# infer size
img0 = cv2.imread(str(imgs[0]))
h, w = img0.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter(str(OUT_MP4), fourcc, FPS, (w, h))
assert vw.isOpened(), f"Could not create: {OUT_MP4}"

for i, p in enumerate(imgs, 1):
    im = cv2.imread(str(p))
    if im is None:
        continue
    if im.shape[1] != w or im.shape[0] != h:
        im = cv2.resize(im, (w, h))
    vw.write(im)
    if i % 200 == 0 or i == len(imgs):
        print(f"[stitch] {i}/{len(imgs)} frames")

vw.release()
print(f"[stitch] video saved -> {OUT_MP4}")


[stitch] 200/800 frames
[stitch] 400/800 frames
[stitch] 600/800 frames
[stitch] 800/800 frames
[stitch] video saved -> d:\omg2\datasets\APPLE\detections_with_depth_bold_from_frames.mp4


In [6]:
# %% [Patch torch._utils so YOLO can load weights]

import importlib
import types

try:
    import torch
    # Import the submodule explicitly
    _u = importlib.import_module("torch._utils")
    # Attach to torch if missing
    if not hasattr(torch, "_utils"):
        setattr(torch, "_utils", _u)
    # Some builds miss _element_size; define a compatible fallback
    if not hasattr(torch._utils, "_element_size"):
        def _element_size(dtype):
            # element_size() gives bytes per element for a given dtype
            return torch.empty(0, dtype=dtype).element_size()
        torch._utils._element_size = _element_size

    print("✅ torch._utils patched OK")
    print("Torch version:", torch.__version__)
    print("torch._utils file:", _u.__file__)
except Exception as e:
    print("❌ Patch failed:", repr(e))


✅ torch._utils patched OK
Torch version: 2.6.0+cu124
torch._utils file: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\.venv\Lib\site-packages\torch\_utils.py


In [7]:
# %% [Validate and print accuracy metrics + F1]

from pathlib import Path
import csv
from ultralytics import YOLO
import torch

ROOT = Path.cwd()

def pick_weights():
    # Prefer best.pt, then last.pt, newest first, anywhere under project
    cands = []
    for pat in ("**/weights/best.pt", "**/weights/last.pt"):
        for f in ROOT.rglob(pat):
            try:
                cands.append((f, f.stat().st_mtime, 1 if f.name.lower() == "best.pt" else 0))
            except FileNotFoundError:
                pass
    if not cands:
        return None
    cands.sort(key=lambda t: (t[2], t[1]), reverse=True)
    return cands[0][0]

def pick_data_yaml():
    # Common spots first
    for y in [
        ROOT / "datasets" / "APPLE" / "data.yaml",
        ROOT / "dataset"  / "APPLE" / "data.yaml",
        ROOT / "data.yaml",
    ]:
        if y.exists():
            return y
    # Heuristic search
    for y in ROOT.rglob("data.yaml"):
        try:
            txt = y.read_text(encoding="utf-8", errors="ignore")
            if "train:" in txt and "names:" in txt:
                return y
        except Exception:
            pass
    return None

def read_latest_results_csv():
    # Fallback if val cannot run
    csvs = sorted(ROOT.rglob("results.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        return None
    with open(csvs[0], newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return None
    last = rows[-1]
    def fv(k, alt=None):
        v = last.get(k) if last.get(k) not in (None, "") else (last.get(alt) if alt else None)
        try:
            return float(v) if v not in (None, "") else None
        except Exception:
            return None
    return {
        "P": fv("metrics/precision(B)", "metrics/precision"),
        "R": fv("metrics/recall(B)",    "metrics/recall"),
        "mAP50": fv("metrics/mAP50(B)", "metrics/mAP50"),
        "mAP50-95": fv("metrics/mAP50-95(B)", "metrics/mAP50-95"),
        "source": str(csvs[0]),
    }

weights_path = pick_weights()
if not weights_path:
    raise FileNotFoundError("No weights found anywhere under this project (looked for **/weights/best.pt or last.pt).")

print(f"[info] Using weights: {weights_path}")
data_yaml = pick_data_yaml()
print(f"[info] Using data.yaml: {data_yaml if data_yaml else '(none — will use weights-embedded args if available)'}")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"[info] Device: {device}")

try:
    model = YOLO(str(weights_path))
    val_batch = 8 if device == "cuda" else 1
    results = model.val(
        data=str(data_yaml) if data_yaml else None,
        device=device,
        imgsz=640,
        batch=val_batch,
        iou=0.6,
        conf=0.001
    )
    d = getattr(results, "results_dict", {}) or {}
    P   = d.get("metrics/precision(B)", d.get("metrics/precision"))
    R   = d.get("metrics/recall(B)",    d.get("metrics/recall"))
    m50 = d.get("metrics/mAP50(B)",     d.get("metrics/mAP50"))
    m95 = d.get("metrics/mAP50-95(B)",  d.get("metrics/mAP50-95"))
    F1  = (2*P*R/(P+R)) if (P is not None and R is not None and (P+R)>0) else None

    print("\n=== VALIDATION METRICS (fresh) ===")
    print(f"Precision     : {P}")
    print(f"Recall        : {R}")
    print(f"F1-score      : {F1}")
    print(f"mAP@0.5       : {m50}")
    print(f"mAP@0.5:0.95  : {m95}")

except Exception as e:
    print("\n[warn] model.val() failed:", repr(e))
    m = read_latest_results_csv()
    if not m:
        raise SystemExit("No results.csv found either. Re-run training/val after fixing PyTorch.")
    F1 = (2*m['P']*m['R']/(m['P']+m['R'])) if (m['P'] is not None and m['R'] is not None and (m['P']+m['R'])>0) else None
    print("\n=== LAST-EPOCH METRICS (from results.csv) ===")
    print(f"source        : {m['source']}")
    print(f"Precision     : {m['P']}")
    print(f"Recall        : {m['R']}")
    print(f"F1-score      : {F1}")
    print(f"mAP@0.5       : {m['mAP50']}")
    print(f"mAP@0.5:0.95  : {m['mAP50-95']}")


[info] Using weights: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv4\weights\best.pt
[info] Using data.yaml: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\data.yaml
[info] Device: cuda

[warn] model.val() failed: AttributeError("module 'torch' has no attribute 'dtype'")

=== LAST-EPOCH METRICS (from results.csv) ===
source        : e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv4\results.csv
Precision     : 0.71137
Recall        : 0.75267
F1-score      : 0.7314374715171716
mAP@0.5       : 0.79805
mAP@0.5:0.95  : 0.6405


In [8]:
# %% [TEMP PATCH] Make torch._utils and torch.dtype available

import importlib
import torch

# Ensure torch._utils exists and has _element_size
try:
    u = importlib.import_module("torch._utils")
    if not hasattr(torch, "_utils"):
        torch._utils = u
    if not hasattr(torch._utils, "_element_size"):
        def _element_size(dtype):
            return torch.empty(0, dtype=dtype).element_size()
        torch._utils._element_size = _element_size
except Exception as e:
    print("torch._utils patch failed:", repr(e))

# Ensure torch.dtype symbol exists (some broken builds miss it)
if not hasattr(torch, "dtype"):
    torch.dtype = type(torch.float32)

print("✅ Torch patched. Has _utils?", hasattr(torch, "_utils"), "| Has dtype?", hasattr(torch, "dtype"))


✅ Torch patched. Has _utils? True | Has dtype? True


In [ ]:
# %% [ACCURACY SUMMARY] Read latest Ultralytics results.csv and print P, R, F1, mAPs

from pathlib import Path
import csv, math

ROOT = Path.cwd()

def scan_results_csv():
    # find all results.csv under any "runs" folder in project
    csvs = []
    seen = set()
    for runs_dir in ROOT.rglob("runs"):
        if runs_dir.is_dir() and str(runs_dir) not in seen:
            seen.add(str(runs_dir))
            csvs.extend(runs_dir.rglob("results.csv"))
    return sorted(set(csvs), key=lambda p: p.stat().st_mtime, reverse=True)

def last_row(csv_path):
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
    return rows[-1] if rows else None

def f_get(row, k, alt=None):
    if row is None: return None
    v = row.get(k) if row.get(k) not in (None, "") else (row.get(alt) if alt else None)
    try:
        return float(v) if v not in (None, "") else None
    except Exception:
        return None

def f1(p, r):
    return (2*p*r/(p+r)) if (p is not None and r is not None and (p+r)>0) else None

def fmt(x):
    return f"{x:.4f}" if isinstance(x, (int, float)) and not math.isnan(x) else "NA"

rows = []
for c in scan_results_csv():
    r = last_row(c)
    if not r: 
        continue
    P  = f_get(r, "metrics/precision(B)", "metrics/precision")
    R  = f_get(r, "metrics/recall(B)",    "metrics/recall")
    M5 = f_get(r, "metrics/mAP50(B)",     "metrics/mAP50")
    M9 = f_get(r, "metrics/mAP50-95(B)",  "metrics/mAP50-95")
    rows.append({
        "run_name": c.parent.name,
        "csv": str(c),
        "P": P, "R": R, "F1": f1(P, R), "mAP50": M5, "mAP50-95": M9
    })

print("=== Accuracy Summary (latest results.csv first) ===")
for i, r in enumerate(rows[:10], 1):
    print(f"{i:>2}. {r['run_name']}")
    print(f"    P={fmt(r['P'])}  R={fmt(r['R'])}  F1={fmt(r['F1'])}  mAP@0.5={fmt(r['mAP50'])}  mAP@0.5:0.95={fmt(r['mAP50-95'])}")
    print(f"    file: {r['csv']}")


=== Accuracy Summary (latest results.csv first) ===
 1. deepHarvestNet-aam-ebca-fm-kwconv4
    P=0.7114  R=0.7527  F1=0.7314  mAP@0.5=0.7981  mAP@0.5:0.95=0.6405
    file: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv4\results.csv
 2. deepHarvestNet-aam-ebca-fm-kwconv-fixed
    P=0.7860  R=0.7929  F1=0.7894  mAP@0.5=0.8697  mAP@0.5:0.95=0.7343
    file: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv-fixed\results.csv
 3. deepHarvestNet-aam-ebca-fm-kwconv32
    P=0.0000  R=0.0000  F1=NA  mAP@0.5=0.0000  mAP@0.5:0.95=0.0000
    file: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv32\results.csv
 4. deepHarvestNet-aam-ebca-fm-kwconv3
    P=0.0000  R=0.0000  F1=NA  mAP@0.5=0.0000  mAP@0.5:0.95=0.0000
    file: e:\dection\AAM, EBCA, FM, KWCo AdaBins-style depth\datasets\APPLE\runs\deepHarvestNet-aam-ebca-fm-kwconv3\resu

: 